# Reorder Prediction — DESD Local Food Marketplace

Predicts which products a customer is likely to reorder in the next 30 days.

**Feature engineering**: RFM per (customer, product) pair

| Feature | Meaning |
|---|---|
| **Recency** | Days since last ordered this product |
| **Frequency** | Number of distinct orders containing this product |
| **Monetary** | Total spend on this product |
| **Interval** | Average days between orders of this product |

**Model**: Logistic Regression trained on binary label `will_reorder` (ordered within next 30 days)

Output: `reorder_model.pkl` + `reorder_metrics.json`

**Author: Nazli**

In [1]:
import os, sys, json, warnings, pathlib
warnings.filterwarnings('ignore')

sys.path.insert(0, str(pathlib.Path('__file__').resolve().parent.parent.parent))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'backend.settings')
try:
    import django
    django.setup()
    DJANGO_OK = True
except Exception as e:
    print(f'[warn] Django not available: {e}. Using synthetic data only.')
    DJANGO_OK = False

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
import joblib

OUT_DIR    = pathlib.Path('.')
MODEL_PATH = OUT_DIR / 'reorder_model.pkl'
METRICS_PATH = OUT_DIR / 'reorder_metrics.json'
print('Setup complete.')

[warn] Django not available: No module named 'backend'. Using synthetic data only.


Setup complete.


## 1  Load customer order history from DB

In [2]:
def load_db_data() -> pd.DataFrame:
    """Pull all OrderItems with customer context."""
    from api.models import OrderItem
    rows = (
        OrderItem.objects
        .filter(order__delivery_date__isnull=False)
        .select_related('order', 'product', 'order__producer')
        .values(
            'order__customer_name',
            'product__id', 'product__name',
            'order__delivery_date', 'quantity', 'unit_price',
        )
    )
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(list(rows))
    df.columns = ['customer', 'product_id', 'product_name', 'date', 'quantity', 'unit_price']
    df['date'] = pd.to_datetime(df['date'])
    return df

if DJANGO_OK:
    db_df = load_db_data()
    print(f'DB rows: {len(db_df)}')
else:
    db_df = pd.DataFrame()

## 2  Synthetic data (demo when DB is sparse)

In [3]:
def make_synthetic_data(n_customers=15, n_products=10, seed=42) -> pd.DataFrame:
    """
    Generates purchase history where some (customer, product) pairs repeat
    at consistent intervals — these become positive reorder examples.
    """
    rng = np.random.default_rng(seed)
    products = [f'Product_{i}' for i in range(1, n_products + 1)]
    customers = [f'customer_{i}@example.com' for i in range(1, n_customers + 1)]
    rows = []
    start = pd.Timestamp('2023-01-01')
    for cust in customers:
        # Each customer has 3-6 favourite products they buy regularly
        n_favs  = rng.integers(3, 7)
        favs    = rng.choice(products, size=n_favs, replace=False)
        for prod in favs:
            # Regular buyer: 4-12 purchases over the year at regular intervals
            n_purchases = rng.integers(4, 13)
            interval_days = rng.integers(14, 45)   # every 2-6 weeks
            for j in range(n_purchases):
                date = start + pd.Timedelta(days=int(j * interval_days + rng.integers(0, 5)))
                qty  = rng.integers(1, 6)
                price = round(rng.uniform(1.0, 8.0), 2)
                rows.append({'customer': cust, 'product_id': int(prod.split('_')[1]),
                             'product_name': prod, 'date': date,
                             'quantity': qty, 'unit_price': price})
        # Occasional one-off purchases of other products (negative examples)
        others = [p for p in products if p not in favs]
        for prod in rng.choice(others, size=min(3, len(others)), replace=False):
            date = start + pd.Timedelta(days=int(rng.integers(0, 300)))
            rows.append({'customer': cust, 'product_id': int(prod.split('_')[1]),
                         'product_name': prod, 'date': date,
                         'quantity': 1, 'unit_price': round(rng.uniform(1.0, 8.0), 2)})
    df = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
    return df

MIN_ROWS = 30
if len(db_df) >= MIN_ROWS:
    df = db_df.copy()
    print(f'Using DB data ({len(df)} rows).')
else:
    df = make_synthetic_data()
    print(f'Using synthetic data ({len(df)} rows).')

print(f"Unique customers: {df['customer'].nunique()}  |  Products: {df['product_id'].nunique()}")

Using synthetic data (604 rows).
Unique customers: 15  |  Products: 10


## 3  RFM feature engineering

For each (customer, product) pair, compute features **as of a snapshot date** and label whether they reordered within the following 30 days.  We slide a window across the timeline to create multiple training examples per pair.

In [4]:
REORDER_WINDOW_DAYS = 30
df['date'] = pd.to_datetime(df['date'])

def build_rfm_features(df: pd.DataFrame, snapshot_date: pd.Timestamp) -> pd.DataFrame:
    """Compute RFM features for all (customer, product) pairs visible before snapshot_date."""
    past   = df[df['date'] <= snapshot_date].copy()
    future = df[(df['date'] > snapshot_date) &
                (df['date'] <= snapshot_date + pd.Timedelta(days=REORDER_WINDOW_DAYS))]
    reorder_set = set(zip(future['customer'], future['product_id']))

    grp = past.groupby(['customer', 'product_id'])
    feat = grp.agg(
        recency  = ('date', lambda x: (snapshot_date - x.max()).days),
        frequency= ('date', 'count'),
        monetary = ('unit_price', lambda x: (x * past.loc[x.index, 'quantity']).sum()),
        last_qty = ('quantity', 'last'),
    ).reset_index()
    feat['will_reorder'] = feat.apply(
        lambda r: int((r['customer'], r['product_id']) in reorder_set), axis=1
    )
    # Average inter-purchase interval in days
    def avg_interval(dates):
        s = sorted(dates)
        if len(s) < 2:
            return 999
        diffs = [(s[i+1] - s[i]).days for i in range(len(s)-1)]
        return np.mean(diffs)

    intervals = past.groupby(['customer', 'product_id'])['date'].apply(avg_interval).reset_index()
    intervals.columns = ['customer', 'product_id', 'avg_interval_days']
    feat = feat.merge(intervals, on=['customer', 'product_id'])
    return feat

# Slide snapshot date every 30 days across the dataset timeline
min_date = df['date'].min() + pd.Timedelta(days=60)   # need at least 2 months of history
max_date = df['date'].max() - pd.Timedelta(days=REORDER_WINDOW_DAYS)
all_features = []
for snap in pd.date_range(min_date, max_date, freq='30D'):
    all_features.append(build_rfm_features(df, snap))

feat_df = pd.concat(all_features, ignore_index=True)
print(f'Total training samples: {len(feat_df)}')
print(f'Reorder rate: {feat_df["will_reorder"].mean():.1%}')
print(feat_df[['recency','frequency','monetary','avg_interval_days','will_reorder']].describe().round(2))

Total training samples: 1347
Reorder rate: 26.1%
       recency  frequency  monetary  avg_interval_days  will_reorder
count  1347.00    1347.00   1347.00            1347.00       1347.00
mean    109.34       4.52     57.08             344.86          0.26
std     102.49       3.29     49.08             453.52          0.44
min       0.00       1.00      1.14              13.75          0.00
25%      21.00       1.00      6.45              29.00          0.00
50%      77.00       4.00     50.98              36.44          0.00
75%     181.50       7.00     92.42             999.00          1.00
max     418.00      12.00    186.66             999.00          1.00


## 4  Train Logistic Regression

In [5]:
FEATURE_COLS = ['recency', 'frequency', 'monetary', 'last_qty', 'avg_interval_days']
X = feat_df[FEATURE_COLS].fillna(0).values
y = feat_df['will_reorder'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
])
pipeline.fit(X_train, y_train)
print('Training done.')

y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No reorder', 'Will reorder']))
auc = roc_auc_score(y_test, y_proba)
print(f'AUC-ROC: {auc:.4f}')

Training done.
              precision    recall  f1-score   support

  No reorder       0.99      0.85      0.92       200
Will reorder       0.70      0.99      0.82        70

    accuracy                           0.89       270
   macro avg       0.85      0.92      0.87       270
weighted avg       0.92      0.89      0.89       270

AUC-ROC: 0.9417


## 5  Feature importance

In [6]:
coefs = pipeline.named_steps['clf'].coef_[0]
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.barh(FEATURE_COLS, coefs, color=['#16a34a' if c > 0 else '#dc2626' for c in coefs])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Logistic regression coefficient')
ax.set_title('Feature importance — Reorder Prediction', fontweight='bold')
fig.tight_layout()
fig.savefig(OUT_DIR / 'reorder_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Chart saved.')

Chart saved.


## 6  Export model + item index

In [7]:
# Build per-customer purchase profile for inference (pre-compute RFM per pair)
latest_snapshot = df['date'].max()
customer_profiles = build_rfm_features(df, latest_snapshot)

artifact = {
    'pipeline':          pipeline,
    'feature_cols':      FEATURE_COLS,
    'customer_profiles': customer_profiles,  # pre-computed for fast lookup
    'reorder_window_days': REORDER_WINDOW_DAYS,
}
joblib.dump(artifact, MODEL_PATH)
print(f'Model saved → {MODEL_PATH}')

from sklearn.metrics import precision_score, recall_score, f1_score
metrics = {
    'generated_at': pd.Timestamp.now().isoformat(),
    'n_training_samples': int(len(X_train)),
    'n_test_samples':     int(len(X_test)),
    'reorder_rate':       round(float(y.mean()), 4),
    'auc_roc':   round(float(auc), 4),
    'precision': round(float(precision_score(y_test, y_pred)), 4),
    'recall':    round(float(recall_score(y_test, y_pred)), 4),
    'f1_score':  round(float(f1_score(y_test, y_pred)), 4),
    'feature_cols': FEATURE_COLS,
}
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Metrics saved → {METRICS_PATH}')
print(json.dumps(metrics, indent=2))

Model saved → reorder_model.pkl
Metrics saved → reorder_metrics.json
{
  "generated_at": "2026-04-22T19:23:34.641012",
  "n_training_samples": 1077,
  "n_test_samples": 270,
  "reorder_rate": 0.2606,
  "auc_roc": 0.9417,
  "precision": 0.697,
  "recall": 0.9857,
  "f1_score": 0.8166,
  "feature_cols": [
    "recency",
    "frequency",
    "monetary",
    "last_qty",
    "avg_interval_days"
  ]
}
